# MPC-PZP Project – Word statistics: CPU vs. multi-core CPU vs. GPU vs. Apache Spark

**Task:** load `data.txt` and `stop_words.txt`, process `data.txt` word by word, keep only words with
**4 ≤ length ≤ 8** (words shorter than 4 or longer than 8 characters are filtered out), remove the stop words and compute

* the **most frequent** word and its number of occurrences,
* the **least frequent** word and its number of occurrences,
* the **total number of words** left after filtering.

The same task is solved by 4 algorithms and their processing times are compared in a pie ("cake") chart:

| # | Version | Technology |
|---|---------|------------|
| 1 | CPU – single-threaded | pure Python, `collections.Counter` |
| 2 | CPU – multi-core | `multiprocessing.Pool`, one worker per CPU core |
| 3 | GPU | dictionary word → id, `int32` vectors, custom CUDA kernels via **CuPy** |
| 4 | Apache Spark | PySpark RDD API, `local[*]` = all CPU cores |

### How to run in Google Colab
1. **Runtime → Change runtime type → T4 GPU** (needed for version 3).
2. **Runtime → Run all**. If the input files are not found, the notebook asks you to upload
   `data.txt` and `stop_words.txt` (you can also drag them into the *Files* panel on the left).

### Design decisions (common to all versions, so the results are identical)
* **Tokenisation:** a word is a sequence of letters `a–z`, optionally joined by an inner hyphen or apostrophe
  (`summer-house`, `whale's`). Everything else (digits, punctuation, `--`) separates words.
* **Case-insensitive:** text and stop words are lower-cased (`The` and `the` are the same word).
* **Ties:** many words share the same count (especially count 1). The alphabetically first word is reported and
  the number of tied words is printed as well.
* **Timing:** every algorithm is timed end-to-end *including reading the files*; one untimed warm-up run is
  done first (CUDA kernel compilation, JVM warm-up, OS file cache) and the mean of `REPEATS` runs is reported.
  One-time infrastructure start-up (Spark session, CUDA context, process pool) is excluded for all versions.

## 0. Environment setup

In [ ]:
# Colab already contains NumPy, Matplotlib and CuPy (on GPU runtimes).
# PySpark 3.5.x works with Java 8/11/17; PySpark 4.x needs Java >= 17.
import importlib.metadata, re, shutil, subprocess, sys

def java_major():
    if shutil.which("java") is None:
        return None
    out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    m = re.search(r'version "(\d+)(?:\.(\d+))?', out)
    if not m:
        return None
    major = int(m.group(1))
    return int(m.group(2)) if major == 1 else major          # "1.8" -> 8

def pyspark_major():
    try:
        return int(importlib.metadata.version("pyspark").split(".")[0])
    except importlib.metadata.PackageNotFoundError:
        return None

if java_major() is None:
    print("Installing Java 17 ...")
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null",
                   shell=True, check=True)

if pyspark_major() is None or (pyspark_major() >= 4 and java_major() < 17):
    print("Installing PySpark 3.5.x ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark>=3.5,<4"], check=True)

print("Java    :", java_major())
print("PySpark :", importlib.metadata.version("pyspark"))

## 1. Input files
The cell looks for `data.txt` and `stop_words.txt` in `./data`, `./pzp-project/data` or `.`. If they are not there it
tries to clone the project repository and, if that is not possible, opens an upload dialog.

`DATA_SCALE` can be set to e.g. `20` to replicate `data.txt` 20× – with a bigger workload the advantage of the
parallel versions becomes much more visible (the original file is only ~1.2 MB).

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/ixargamessvk/pzp-project.git"   # used only if the repository is public
DATA_SCALE = 1                                                 # >1 => data.txt is replicated DATA_SCALE times

def find_data_dir():
    for d in ("data", os.path.join("pzp-project", "data"), "."):
        if os.path.isfile(os.path.join(d, "data.txt")) and os.path.isfile(os.path.join(d, "stop_words.txt")):
            return d
    return None

DATA_DIR = find_data_dir()
if DATA_DIR is None:                                # 1) try to clone the repository
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, "pzp-project"],
                   capture_output=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    DATA_DIR = find_data_dir()
if DATA_DIR is None:                                # 2) ask the user to upload the files
    from google.colab import files
    print("Please upload data.txt and stop_words.txt")
    os.makedirs("data", exist_ok=True)
    for name, content in files.upload().items():
        target = "stop_words.txt" if "stop" in name.lower() else "data.txt"
        with open(os.path.join("data", target), "wb") as f:
            f.write(content)
    DATA_DIR = find_data_dir()
assert DATA_DIR is not None, "data.txt / stop_words.txt not found"

DATA_PATH = os.path.abspath(os.path.join(DATA_DIR, "data.txt"))
STOP_PATH = os.path.abspath(os.path.join(DATA_DIR, "stop_words.txt"))

if DATA_SCALE > 1:
    with open(DATA_PATH, encoding="utf-8-sig") as src:
        content = src.read()
    scaled_path = os.path.abspath(f"data_x{DATA_SCALE}.txt")
    with open(scaled_path, "w", encoding="utf-8") as dst:
        for _ in range(DATA_SCALE):
            dst.write(content)
            dst.write("\n")
    DATA_PATH = scaled_path

print(f"data.txt       : {DATA_PATH}  ({os.path.getsize(DATA_PATH) / 1e6:.2f} MB)")
print(f"stop_words.txt : {STOP_PATH}")

## 2. Shared configuration and helper functions
These helpers are used by all versions, which guarantees that every algorithm applies exactly the same
tokenisation, filtering and statistics rules.

In [ ]:
import os, re, time
from collections import Counter

MIN_LEN, MAX_LEN = 4, 8          # keep 4 <= len(word) <= 8, i.e. filter out len < 4 and len > 8
CPU_CORES = os.cpu_count() or 1  # all available CPU cores
REPEATS = 3                      # timed runs per algorithm (after one untimed warm-up run)

# word = letters, optionally joined by an inner hyphen / apostrophe ("summer-house", "whale's")
WORD_RE = re.compile(r"[a-z]+(?:['-][a-z]+)*")


def read_text(path):
    """Load a whole text file (utf-8-sig also strips the BOM at the start of data.txt)."""
    with open(path, encoding="utf-8-sig") as f:
        return f.read()


def tokenize(text):
    """Split text into lower-cased words."""
    return WORD_RE.findall(text.lower())


def load_stop_words(path):
    """Stop words as a frozenset -> O(1) membership test."""
    return frozenset(tokenize(read_text(path)))


def compute_stats(counts):
    """counts: mapping word -> occurrences (already filtered). Ties are broken alphabetically."""
    if not counts:
        return {"total_words": 0, "unique_words": 0, "most_frequent": None,
                "least_frequent": None, "least_frequent_ties": 0}
    most = min(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    least = min(counts.items(), key=lambda kv: (kv[1], kv[0]))
    return {
        "total_words": sum(counts.values()),
        "unique_words": len(counts),
        "most_frequent": tuple(most),
        "least_frequent": tuple(least),
        "least_frequent_ties": sum(1 for c in counts.values() if c == least[1]),
    }


RESULTS, TIMINGS = {}, {}        # name -> statistics / mean processing time [s]


def print_results(name, stats, mean_time, times):
    print(f"===== {name} =====")
    print(f"  Number of words after filtering : {stats['total_words']:,}")
    print(f"  Number of distinct words        : {stats['unique_words']:,}")
    if stats["most_frequent"]:
        w, c = stats["most_frequent"]
        print(f"  Most frequent word              : '{w}'  ->  {c:,} occurrences")
        w, c = stats["least_frequent"]
        print(f"  Least frequent word             : '{w}'  ->  {c:,} occurrence(s)"
              f"   ({stats['least_frequent_ties']:,} words share this count, alphabetically first shown)")
    runs = ", ".join(f"{t * 1000:.1f}" for t in times)
    print(f"  Processing time                 : {mean_time * 1000:.1f} ms   (mean of runs: {runs} ms)")
    for phase, t in stats.get("phases", {}).items():
        print(f"      - {phase:<52}: {t * 1000:8.2f} ms")
    print()


def run_benchmark(name, fn, *args, repeats=REPEATS):
    """One untimed warm-up run, then `repeats` timed runs; stores and prints the results."""
    fn(*args)
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        stats = fn(*args)
        times.append(time.perf_counter() - t0)
    mean_time = sum(times) / len(times)
    RESULTS[name], TIMINGS[name] = stats, mean_time
    print_results(name, stats, mean_time, times)
    return stats

## 3. Version 1 – CPU, single-threaded
Single pass over the word stream: tokenise → length filter → stop-word filter (hash-set lookup, O(1)) → count with
`collections.Counter` (hash map, C implementation). Overall complexity **O(N)** in the number of words.

In [ ]:
def count_words(text, stop_words):
    """Process the text word by word: length filter, stop-word filter, counting."""
    lo, hi = MIN_LEN, MAX_LEN
    return Counter(w for w in tokenize(text) if lo <= len(w) <= hi and w not in stop_words)


def cpu_single(data_path, stop_path):
    stop_words = load_stop_words(stop_path)
    text = read_text(data_path)
    return compute_stats(count_words(text, stop_words))

In [ ]:
run_benchmark("CPU 1 thread", cpu_single, DATA_PATH, STOP_PATH);

## 4. Version 2 – CPU, multi-core
Data parallelism (map-reduce):
1. `data.txt` is split into as many equally sized chunks as there are CPU cores (cuts are made only at line breaks,
   so no word is split),
2. every worker counts the filtered words of its chunk (**map**) – the same `count_words` as version 1,
3. the partial `Counter`s are merged (**reduce**) and the statistics are computed.

> **Why processes and not threads?** CPython has the *Global Interpreter Lock* – only one thread can execute Python
> byte-code at a time, so CPU-bound work does not get faster with `threading`. `multiprocessing` starts one
> interpreter per core, which really uses **all CPU cores** in parallel.

In [ ]:
import multiprocessing as mp


def split_text(text, n_parts):
    """Split text into n_parts chunks of ~equal size, cutting only at line breaks."""
    step = max(1, len(text) // n_parts)
    chunks, start = [], 0
    for _ in range(n_parts - 1):
        cut = text.find("\n", start + step)
        if cut == -1:
            break
        chunks.append(text[start:cut])
        start = cut
    chunks.append(text[start:])
    return chunks


def _count_chunk(args):
    """Worker (map step) – runs in a separate process."""
    chunk, stop_words = args
    return count_words(chunk, stop_words)


def make_pool(n_workers=CPU_CORES):
    """Process pool; 'fork' (Linux / Colab) makes starting the workers cheap."""
    methods = mp.get_all_start_methods()
    ctx = mp.get_context("fork") if "fork" in methods else mp.get_context()
    return ctx.Pool(processes=n_workers)


def cpu_multi(data_path, stop_path, pool, n_workers=CPU_CORES):
    stop_words = load_stop_words(stop_path)
    text = read_text(data_path)
    chunks = split_text(text, n_workers)
    partial = pool.map(_count_chunk, [(chunk, stop_words) for chunk in chunks])   # map (parallel)
    total = partial[0]
    for c in partial[1:]:                                                         # reduce
        total.update(c)
    return compute_stats(total)

In [ ]:
print(f"Available CPU cores: {CPU_CORES}\n")
POOL = make_pool(CPU_CORES)          # created once and reused (infrastructure start-up is not timed)
run_benchmark(f"CPU {CPU_CORES} cores", cpu_multi, DATA_PATH, STOP_PATH, POOL, CPU_CORES);

## 5. Version 3 – GPU (CuPy + custom CUDA kernels)
A GPU cannot work with Python strings, so the data are converted to numbers first.

**CPU part (pre-processing)**
1. load and tokenise both files,
2. **length filter (4–8 characters) on the CPU**,
3. build a **dictionary `word → id`** from all words of both files (ids are given in alphabetical order, so the
   smallest id also means the alphabetically first word → same tie-breaking as the CPU versions),
4. convert `data.txt` and `stop_words.txt` into **`int32` vectors of ids**.

**GPU part**
1. copy both integer vectors to the GPU,
2. kernel `mark_stop_words` – one thread per stop word sets `is_stop[id] = 1` (lookup table of size |dictionary|),
3. kernel `filter_and_count` – **stop-word filtering on the GPU**: each thread takes one word id, drops it if
   `is_stop[id]`, otherwise increments `counts[id]` with `atomicAdd` (filter + histogram fused in one pass,
   grid-stride loop),
4. statistics on the GPU (`sum`, `max`, `min` of non-zero counts, `count_nonzero`); only a few scalars are copied
   back to the host.

In [ ]:
import numpy as np

try:
    import cupy as cp
    GPU_AVAILABLE = cp.cuda.runtime.getDeviceCount() > 0
except Exception as exc:                       # no CuPy or no CUDA device
    cp, GPU_AVAILABLE = None, False
    print("GPU version not available:", exc)

CUDA_SRC = r"""
extern "C" __global__
void mark_stop_words(const int* __restrict__ stop_ids, const int n_stop,
                     unsigned char* __restrict__ is_stop)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n_stop) is_stop[stop_ids[i]] = 1;
}

extern "C" __global__
void filter_and_count(const int* __restrict__ word_ids, const long long n_words,
                      const unsigned char* __restrict__ is_stop, int* __restrict__ counts)
{
    long long i = (long long)blockIdx.x * blockDim.x + threadIdx.x;
    const long long stride = (long long)gridDim.x * blockDim.x;
    for (; i < n_words; i += stride) {          // grid-stride loop
        const int id = word_ids[i];
        if (!is_stop[id])                        // stop-word filter
            atomicAdd(&counts[id], 1);           // histogram
    }
}
"""

THREADS_PER_BLOCK = 256
MAX_BLOCKS = 65535

if GPU_AVAILABLE:
    try:                                       # compile the CUDA kernels (NVRTC, cached after the first run)
        _cuda_module = cp.RawModule(code=CUDA_SRC)
        mark_stop_kernel = _cuda_module.get_function("mark_stop_words")
        filter_count_kernel = _cuda_module.get_function("filter_and_count")
    except Exception as exc:
        GPU_AVAILABLE = False
        print("CUDA kernel compilation failed:", exc)


def gpu_preprocess(data_path, stop_path):
    """CPU part: load, tokenise, LENGTH filter, dictionary word->id, encode to int32 vectors."""
    lo, hi = MIN_LEN, MAX_LEN
    words = [w for w in tokenize(read_text(data_path)) if lo <= len(w) <= hi]
    stop_words = load_stop_words(stop_path)
    vocab = sorted(set(words) | stop_words)                 # id order == alphabetical order
    word_to_id = {w: i for i, w in enumerate(vocab)}
    word_ids = np.fromiter(map(word_to_id.__getitem__, words), dtype=np.int32, count=len(words))
    stop_ids = np.fromiter(map(word_to_id.__getitem__, stop_words), dtype=np.int32, count=len(stop_words))
    return vocab, word_ids, stop_ids


def gpu_version(data_path, stop_path):
    sync = cp.cuda.Stream.null.synchronize
    t0 = time.perf_counter()
    vocab, word_ids, stop_ids = gpu_preprocess(data_path, stop_path)
    t1 = time.perf_counter()

    d_words = cp.asarray(word_ids)                          # host -> device
    d_stop = cp.asarray(stop_ids)
    sync()
    t2 = time.perf_counter()

    n_vocab, n_words, n_stop = len(vocab), int(word_ids.size), int(stop_ids.size)
    d_is_stop = cp.zeros(n_vocab, dtype=cp.uint8)
    d_counts = cp.zeros(n_vocab, dtype=cp.int32)
    tpb = THREADS_PER_BLOCK
    if n_stop:
        mark_stop_kernel(((n_stop + tpb - 1) // tpb,), (tpb,), (d_stop, np.int32(n_stop), d_is_stop))
    if n_words:
        blocks = min((n_words + tpb - 1) // tpb, MAX_BLOCKS)
        filter_count_kernel((blocks,), (tpb,), (d_words, np.int64(n_words), d_is_stop, d_counts))

    # statistics computed on the GPU
    d_total = d_counts.sum(dtype=cp.int64)
    d_unique = cp.count_nonzero(d_counts)
    d_max = d_counts.max()
    d_min = cp.where(d_counts > 0, d_counts, np.iinfo(np.int32).max).min()
    d_max_ids = cp.flatnonzero(d_counts == d_max)           # sorted ids -> [0] = alphabetically first
    d_min_ids = cp.flatnonzero(d_counts == d_min)
    sync()
    t3 = time.perf_counter()

    total, unique = int(d_total), int(d_unique)             # device -> host (only scalars)
    max_c, min_c = int(d_max), int(d_min)
    max_id, min_id, ties = int(d_max_ids[0]), int(d_min_ids[0]), int(d_min_ids.size)
    t4 = time.perf_counter()

    if total == 0:
        return compute_stats({})
    return {
        "total_words": total,
        "unique_words": unique,
        "most_frequent": (vocab[max_id], max_c),
        "least_frequent": (vocab[min_id], min_c),
        "least_frequent_ties": ties,
        "phases": {
            "CPU: load, tokenise, length filter, dictionary, encode": t1 - t0,
            "Host -> Device copy": t2 - t1,
            "GPU: stop-word filter + counting + statistics": t3 - t2,
            "Device -> Host copy of results": t4 - t3,
        },
    }

In [ ]:
if GPU_AVAILABLE:
    props = cp.cuda.runtime.getDeviceProperties(0)
    print("GPU:", props["name"].decode(), "\n")

    # show what the GPU actually receives
    vocab, word_ids, stop_ids = gpu_preprocess(DATA_PATH, STOP_PATH)
    print(f"Dictionary size        : {len(vocab):,} words")
    print(f"Encoded data vector    : {word_ids.size:,} x int32  ->  {word_ids[:10].tolist()} ...")
    print(f"  = words              : {[vocab[i] for i in word_ids[:10]]} ...")
    print(f"Encoded stop-word vec. : {stop_ids.tolist()}  ->  {[vocab[i] for i in stop_ids]}\n")

    run_benchmark("GPU (CuPy)", gpu_version, DATA_PATH, STOP_PATH);
else:
    print("No CUDA GPU available -> GPU version skipped.")
    print("In Colab select: Runtime -> Change runtime type -> T4 GPU, then run the notebook again.")

## 6. Version 4 – Apache Spark
* Spark runs in local mode with master **`local[*]`** = one executor thread per available CPU core.
* `textFile` reads `data.txt` into an RDD with (at least) one partition per core.
* `mapPartitions` – every partition is processed word by word (length + stop-word filter) and pre-aggregated into a
  local `Counter` (a *combiner*: far fewer records go through the shuffle than with `map(lambda w: (w, 1))`).
* The stop words are sent to the workers once as a **broadcast variable**.
* `reduceByKey` merges the partial counts, the small result (one entry per distinct word) is collected to the driver.

In [ ]:
import operator
from pathlib import Path


def spark_version(data_path, stop_path, sc):
    stop_bc = sc.broadcast(load_stop_words(stop_path))       # sent once to every worker
    word_re, lo, hi = WORD_RE, MIN_LEN, MAX_LEN

    def count_partition(lines):
        stop_words = stop_bc.value
        local = Counter()
        for line in lines:
            for w in word_re.findall(line.lower()):
                if lo <= len(w) <= hi and w not in stop_words:
                    local[w] += 1
        return iter(local.items())

    counts = (sc.textFile(Path(data_path).resolve().as_uri(), minPartitions=sc.defaultParallelism)
                .mapPartitions(count_partition)               # filter + local counting
                .reduceByKey(operator.add)                    # global counting
                .collectAsMap())
    stop_bc.unpersist()
    return compute_stats(counts)

In [ ]:
import sys
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")                                  # use all available CPU cores
         .appName("MPC-PZP word statistics")
         .config("spark.driver.memory", "4g")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print(f"Spark {spark.version}, master = {sc.master}, parallelism = {sc.defaultParallelism} cores\n")

run_benchmark("Apache Spark", spark_version, DATA_PATH, STOP_PATH, sc);

## 7. Summary and verification
All versions must give exactly the same statistics.

In [ ]:
def core(stats):
    return {k: v for k, v in stats.items() if k != "phases"}

names = list(RESULTS)
reference = core(RESULTS[names[0]])
base_time = TIMINGS[names[0]]

print("=" * 78)
print(f"{'Algorithm':<18}{'time [ms]':>12}{'speed-up':>12}   {'most frequent':<18}{'least frequent':<16}{'words':>8}")
print("-" * 78)
for name in names:
    s = RESULTS[name]
    print(f"{name:<18}{TIMINGS[name] * 1000:>12.1f}{base_time / TIMINGS[name]:>11.2f}x   "
          f"{'%s (%d)' % s['most_frequent']:<18}{'%s (%d)' % s['least_frequent']:<16}{s['total_words']:>8,}")
print("=" * 78)
all_same = all(core(RESULTS[n]) == reference for n in names)
print("All versions give identical results:", "YES" if all_same else "NO !!!")
fastest = min(TIMINGS, key=TIMINGS.get)
print(f"Fastest algorithm: {fastest} ({TIMINGS[fastest] * 1000:.1f} ms)")

## 8. Processing times – pie ("cake") chart

In [ ]:
import matplotlib.pyplot as plt

labels = list(TIMINGS)
times_ms = [TIMINGS[n] * 1000 for n in labels]
total_ms = sum(times_ms)
colors = plt.cm.tab10.colors[:len(labels)]

fig, (ax_pie, ax_bar) = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={"width_ratios": [1.1, 1]})

# --- pie ("cake") chart: share of each algorithm in the total processing time ---
wedges, _, _ = ax_pie.pie(
    times_ms,
    colors=colors,
    autopct=lambda p: f"{p:.1f} %" if p >= 4 else "",    # tiny slices are described in the legend
    pctdistance=0.75,
    startangle=90,
    counterclock=False,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 12, "color": "white", "weight": "bold"},
)
ax_pie.legend(wedges,
              [f"{n}: {t:.1f} ms ({100 * t / total_ms:.1f} %)" for n, t in zip(labels, times_ms)],
              title="Algorithm: time (share)", loc="upper center", bbox_to_anchor=(0.5, -0.02),
              fontsize=11, title_fontsize=11)
ax_pie.set_title(f"Processing time – pie chart (mean of {REPEATS} runs)", fontsize=14)
ax_pie.axis("equal")

# --- supplementary bar chart (log scale) – small times are readable here ---
bars = ax_bar.barh(labels, times_ms, color=colors)
ax_bar.set_xscale("log")
ax_bar.invert_yaxis()
ax_bar.bar_label(bars, labels=[f"{t:.1f} ms" for t in times_ms], padding=4, fontsize=11)
ax_bar.set_xlabel("time [ms] (log scale)")
ax_bar.set_xlim(min(times_ms) / 3, max(times_ms) * 5)
ax_bar.set_title("Processing time – bar chart", fontsize=14)
ax_bar.grid(axis="x", which="both", alpha=0.3)

plt.tight_layout()
plt.savefig("processing_times.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Discussion (what to expect)
* `data.txt` is small (~1.2 MB, ~215 000 tokens), so **fixed overheads dominate**: sending chunks to the worker
  processes, starting Spark tasks / Python workers and the shuffle, copying data to the GPU.
* **Multi-core CPU** speeds up the counting, but the speed-up is lower than the number of cores (splitting
  the text, sending the data to the processes and merging the partial results is sequential work – Amdahl's law). A Colab VM
  usually has only 2 cores.
* **GPU:** the kernels themselves take well under a millisecond. Most of the GPU version's time is the **CPU
  pre-processing** (tokenisation, length filter, building the dictionary and encoding), which costs about as much as the whole
  single-threaded algorithm. The GPU pays off only when the encoded data are reused or when the data are much bigger.
* **Apache Spark** is built for distributed big data processing; for a 1 MB file its scheduling, JVM ↔ Python
  serialisation and shuffle overheads make it the slowest version. Its strength is scalability (the same code runs on a cluster
  with terabytes of data).
* Set `DATA_SCALE` (section 1) to e.g. 20–50 and run the notebook again to see how the ratios change for a bigger workload.

In [ ]:
# Clean-up of the infrastructure
if "POOL" in globals():
    POOL.close()
    POOL.join()
if "spark" in globals():
    spark.stop()